# Machine Learning Model Training - PNR Confirmation Predictor

Trains and exports a scikit-learn Gradient Boosting Classifier pipeline for estimating the clearance probability of waitlisted train tickets.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
import joblib

# Load historical data
df = pd.read_csv('../../data_analysis/datasets/historical_bookings.csv')
features = ['waiting_list_number', 'days_to_journey', 'train_type', 'class_type', 'quota', 'is_peak_season']
X = df[features]
y = df['confirmed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training instances: {len(X_train)}, Testing instances: {len(X_test)}')

## 2. Model Pipeline Definition & Training

In [2]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['train_type', 'class_type', 'quota']),
        ('num', 'passthrough', ['waiting_list_number', 'days_to_journey', 'is_peak_season'])
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print('ROC-AUC Score:', round(roc_auc_score(y_test, y_prob), 4))
print(classification_report(y_test, y_pred))

## 3. Save Model Artifact to Disk

In [3]:
joblib.dump(pipeline, 'saved_models/pnr_predictor.joblib')
print('Model successfully persisted to saved_models/pnr_predictor.joblib')